# Lab 1.5 — Structure a Wikipedia article with Gemini

One AI agent receives `X`, the title and body of an English article, and returns `y`, an editable article structure with an infobox where applicable, entity spans and section assignments. The same article runs through zero-shot and few-shot prompts.

**Flow** · X → normalize and number paragraphs → interpret the subject and evidence → infer fields, entities and sections with Gemini → validate JSON → show y and the reconstructed preview.

## Setup

Add `GEMINI_KEY` under Colab **Secrets** and grant notebook access. The setup cell uses a local checkout when available. Before publishing the lab files to GitHub, upload `iis-colab-input.zip` to `/content` using Colab’s **Files** panel. The ZIP should contain `article_agent.py`, `data/`, and `prompts/` at its root.

In [ ]:
%pip -q install google-genai==2.12.1 google-auth==2.49.0

In [ ]:
import json
import subprocess
import sys
from pathlib import Path
from zipfile import ZipFile

ROOT = Path("/content/IIS-nd")
if Path("article_agent.py").is_file():
    ROOT = Path.cwd()
elif Path("../article_agent.py").is_file():
    ROOT = Path.cwd().parent
elif Path("/content/iis-colab-input.zip").is_file() and not (ROOT / "article_agent.py").is_file():
    with ZipFile("/content/iis-colab-input.zip") as archive:
        archive.extractall(ROOT)
elif not ROOT.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/MIM323/IIS-nd.git", str(ROOT)], check=True
    )
if not (ROOT / "article_agent.py").is_file():
    raise FileNotFoundError("Point ROOT to the extracted repository containing article_agent.py.")
sys.path.insert(0, str(ROOT))

from article_agent import MODEL, build_prompt, prepare, preview, read_jsonl, run_agent

instructions = (ROOT / "prompts" / "instructions.txt").read_text(encoding="utf-8")
fixed_examples = read_jsonl(ROOT / "data" / "examples.jsonl")
holdout = read_jsonl(ROOT / "data" / "holdout.jsonl")
target = holdout[0]["X"]
MODEL_ID = MODEL  # Change this if your project uses a different Gemini model.
print("Gemini model:", MODEL_ID)

In [ ]:
from google import genai
from google.colab import userdata

client = genai.Client(api_key=userdata.get("GEMINI_KEY"))

## Data understanding

The code normalizes whitespace and numbers the paragraphs. Nemeris is a fictional country. Its article mentions two cities, so it tests whether the agent distinguishes the current capital from the former one.

In [ ]:
def show_json(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))

print("Input X")
show_json(target)
print("\nPrepared paragraphs")
show_json(prepare(target))

## Reasoning and inference

Both runs use the same instructions and target. Zero-shot sends no examples. Few-shot adds the three labeled pairs from Lab 1.2. In one API call per run, Gemini selects the category and template, assigns headings, extracts facts, and returns the source span for each field. The quotes and section assignments show how to check model decisions against the article.

In [ ]:
runs = {"zero-shot": [], "few-shot": fixed_examples}
outputs = {}
for name, examples in runs.items():
    print(f"\n{name.upper()} PROMPT")
    prompt = build_prompt(instructions, target, examples)
    assert prompt.strip() == (ROOT / "prompts" / f"{name}.txt").read_text(encoding="utf-8").strip()
    print(prompt)
    try:
        outputs[name] = run_agent(client, instructions, target, examples, model=MODEL_ID)
    except ValueError as error:
        print("The model output could not be validated:", error)

## Output generation

Validation checks that the template matches the category, the infobox uses allowed keys, each evidence span appears in the source, and the sections cover every paragraph in order. The preview joins the original paragraphs with the proposed headings. Check the evidence in both outputs and compare the capital with the reference.

In [ ]:
for name, output in outputs.items():
    print(f"\n{name.upper()} RESULT y")
    show_json(output)
    print("\nReview preview")
    print(preview(target, output))

print("\nReference y for inspection")
show_json(holdout[0]["y"])

## Reflection

The agent takes an English article and produces an infobox proposal, an entity list, and a section plan in one request. A structured Gemini response makes the result easy to parse and reuse for an editorial preview. The zero-shot prompt communicates the task through instructions, and the few-shot prompt also shows one worked example for each subject class. The shared input makes differences in their predictions visible. Evidence quotations help an editor inspect the model's field choices, although a quote can contain a value that belongs to a different entity or time period. The Nemeris example tests that distinction with a former capital and a current capital. A larger set of real articles with manually checked labels would help establish when the pipeline improves article structure.